In [21]:
# TP 5 Geoloc 
#Eléonore Veyron

In [22]:

# Echauffement : construire une URL
def url(num,typ,nom):
    return f"https://api-adresse.data.gouv.fr/search/?q={num}+{typ}+{nom},Paris&limit=1"
num = 3
typ = "rue"
nom = "tourelles"
print(url(num,typ,nom))

https://api-adresse.data.gouv.fr/search/?q=3+rue+tourelles,Paris&limit=1


In [23]:
#Echauffement : sélectionner des lignes avec iloc et loc

import pandas as pd

df = pd.DataFrame({
    "number": [3, 18, 7, 12, 5],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
sub = df[df.number > 10]

#1. sub devient df = pd.DataFrame({"number":[18,12],"name":["BERNARDINS","Rivoli"]})
# son index devient 1,3

#2. df.iloc[:2] = ({"number":[3,18],"name":["tourelles","BERNARDIN"]})
# df.loc[:2] = ({"number":[3,18,7],"name":["tourelles","BERNARDIN","Bouvart"]})
print(df.iloc[:2])
print(df.loc[:2])

#3. sub.iloc[:2] = ({"number":[18,12],"name":["BERNARDINS","Rivoli"]})
#sub.loc[:2] = ({"number":[18],"name":["BERNARDINS"]})

#4. sub.iloc[0] = 18, "BERNARDINS"
#sub.loc[0] = eror car pas d'index 0
print(sub.iloc[0])

#5. sub["name"].iloc[0] #on veut la 1ere ligne
#sub["name"].loc[1] #on regarde l'index
print(sub["name"].iloc[0])
print(sub["name"].loc[1]) # ou sub["name"].loc[sub.index[0]]


   number        name
0       3   tourelles
1      18  BERNARDINS
   number        name
0       3   tourelles
1      18  BERNARDINS
2       7     Bouvart
number            18
name      BERNARDINS
Name: 1, dtype: object
BERNARDINS
BERNARDINS


In [ ]:
# Géolocalisation

import pandas as pd

addresses = pd.read_csv('ARTEFACTS-addresses-f6f90e822bcd81e4b53409289753e9ba/data/addresses.csv')
addresses.head(4)

addresses.iloc[:21].to_csv("addresses-small.csv",index=False) #sauvegarde le fichier

In [ ]:
#Une par une 

# on obtient un dictionnaire, c'est toute les rues tourelles de Paris

# avec &limit=1 on obtient seulement les informations pour la 1ere rue de tourelles
#c'est le 1er sous-dico
# requests is the swiss knife for doing http

import requests

def localize_one(num,typ,nom):
    url1 = url(num,typ,nom)
    r = requests.get(url1)
    if 200 <= r.status_code < 300: 
        return r.json()
    else : 
        return None 

import time

t1= time.time()
print(localize_one(18,'rue','BERNARDINS'))
t2= time.time()

print("Faire 30 000 requêtes prendrait", (t2-t1)*30000/3600, "h")
#c'est long car on importe tout le sous-dico

{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [2.350728, 48.850345]}, 'properties': {'label': '18 Rue des Bernardins 75005 Paris', 'score': 0.7544495104895104, 'housenumber': '18', 'id': '75105_0896_00018', 'banId': 'acbebc10-eac0-475f-9466-e7dbe9446b02', 'name': '18 Rue des Bernardins', 'postcode': '75005', 'citycode': '75105', 'x': 652355.3, 'y': 6861340.65, 'city': 'Paris', 'district': 'Paris 5e Arrondissement', 'context': '75, Paris, Île-de-France', 'type': 'housenumber', 'importance': 0.68356, 'depcode': '75', 'street': 'Rue des Bernardins', '_type': 'address'}}], 'query': '18 rue BERNARDINS,Paris'}
Faire 30 000 requêtes prendrait 4.168270031611125 h


In [43]:
#En un seul coup - en python

import requests

import io

def mass_post(filename, col_number, col_type, col_name, col_city):

    with open(filename) as feed:
        response = requests.post(
            "https://api-adresse.data.gouv.fr/search/csv/", 
            files={'data': feed},
            data={'columns': [col_number, col_type, col_name, col_city]})
    if not (200 <= response.status_code < 300):
        print(f"OOPS, got {response.status_code}")
        return None

    result = pd.read_csv(io.StringIO(response.text)) #car pd.read_csvp prend un dataframe

    return result


# votre code

def localize_many(filename, col_number, col_type, col_name, col_city):
    df = pd.read_csv(filename, encoding='utf-8')
    geoloc = mass_post(filename, col_number, col_type, col_name, col_city)
    geoloc = geoloc[['latitude', 'longitude', 'result_city', 'result_type']]
    return df.merge(geoloc, left_index=True, right_index =True)

# try your code on the small sample for starters

addresses_small = localize_many("ARTEFACTS-addresses-f6f90e822bcd81e4b53409289753e9ba/data/addresses-small.csv", "number", "type", "name", "city")
print(addresses_small)
addresses_small = addresses_small[addresses_small['result_city']=='Paris']


   number       type                name   city   latitude  longitude  \
0       5        rue              RIQUET  Paris  48.888134   2.376448   
1     141  boulevard              ITALIE  Paris  46.669103  -1.418872   
2     234        rue  FAUBOURG ST-HONORÉ  Paris  48.876160   2.302766   
3      62        rue        MICHEL BIZOT  Paris  48.835423   2.402040   
4       6        rue                OURS  Paris  48.863311   2.351775   
5      30        rue          PETIT-MUSC  Paris  48.852945   2.364722   
6       3    impasse            MASSONET  Paris  48.895498   2.351580   
7      56        rue            PANOYAUX  Paris  48.867023   2.388347   
8      16        rue             THÉRÈSE  Paris  48.866374   2.335212   
9      11        rue         JEAN ROBERT  Paris  48.889520   2.358170   

        result_city  result_type  
0             Paris  housenumber  
1  La Roche-sur-Yon       street  
2             Paris  housenumber  
3             Paris  housenumber  
4             Paris  